# Sector Positioning Map: Correspondence Analysis

This notebook maps the sector positioning of enterprises with a capitalization strictly above **USD 100 million**.

It performs a genuine correspondence analysis (CA/AFC) on the binary enterprise-by-sector co-occurrence matrix: each cell is `1` when an enterprise is assigned to a sector and `0` otherwise. The AFC uses the chi-square distance between row and column profiles.

The first two AFC dimensions are displayed together:

- enterprise label size is proportional to capitalization;
- sector marker size is proportional to its frequency among selected enterprises.

The notebook reports the total chi-square statistic, eigenvalues, total inertia and the inertia explained by the first two dimensions. It reads `database.db` and writes only analysis exports; it does not modify database records.

In [ ]:
from pathlib import Path
import sqlite3

import numpy as np
import pandas as pd
import plotly.graph_objects as go

ROOT = Path.cwd().resolve()
if not (ROOT / "database.db").exists():
    ROOT = ROOT.parent

DB_PATH = ROOT / "database.db"
EXPORTS_DIR = ROOT / "analyses" / "exports"
CAPITALIZATION_THRESHOLD_MILLIONS = 100.0

if not DB_PATH.exists():
    raise FileNotFoundError(f"Database not found: {DB_PATH}")

EXPORTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Database: {DB_PATH}")
print(f"Capitalization threshold: USD {CAPITALIZATION_THRESHOLD_MILLIONS:,.0f} M")

In [ ]:
def parse_millions(value):
    """Parse a stored amount in USD millions without inventing a value."""
    if value is None or pd.isna(value):
        return None
    text = str(value).strip().replace(" ", "").replace("\u00a0", "").replace("\u202f", "")
    if not text or text.lower() in {"na", "n/a", "null", "none", "unknown"}:
        return None
    if text.count(",") == 1 and text.count(".") == 0:
        text = text.replace(",", ".")
    elif text.count(".") > 1:
        text = text.replace(".", "")
    try:
        return float(text)
    except ValueError:
        return None


def split_sector_labels(value):
    if value is None or pd.isna(value):
        return []
    return sorted({label.strip() for label in str(value).split(",") if label.strip()})


with sqlite3.connect(DB_PATH) as connection:
    companies = pd.read_sql_query(
        """
        SELECT id, name, country, sector, capitalization
        FROM enterprises
        WHERE capitalization IS NOT NULL
          AND TRIM(capitalization) != ''
        """,
        connection,
    )

companies["capitalization_millions"] = companies["capitalization"].map(parse_millions)
companies["sector_labels"] = companies["sector"].map(split_sector_labels)
companies = companies[
    (companies["capitalization_millions"] > CAPITALIZATION_THRESHOLD_MILLIONS)
    & companies["sector_labels"].map(bool)
].copy().sort_values("capitalization_millions", ascending=False).reset_index(drop=True)

print(f"Selected enterprises: {len(companies)}")
print(f"Distinct sectors before CA filtering: {len({label for labels in companies['sector_labels'] for label in labels})}")
display(companies[["name", "capitalization_millions", "country", "sector"]].head(10))

In [ ]:
def min_max_scale(values, minimum, maximum):
    values = np.asarray(values, dtype=float)
    if len(values) == 0 or np.allclose(values.max(), values.min()):
        return np.full(len(values), (minimum + maximum) / 2)
    return minimum + (values - values.min()) / (values.max() - values.min()) * (maximum - minimum)


def correspondence_analysis(cooccurrence):
    """Perform AFC on an enterprise-by-sector co-occurrence matrix.

    The matrix rows are enterprises, columns are sectors and cells indicate an
    observed enterprise-sector co-occurrence. Principal coordinates use the
    chi-square metric; total inertia equals chi2 divided by the matrix total.
    """
    matrix_total = cooccurrence.sum()
    if matrix_total <= 0:
        raise ValueError("The enterprise-by-sector co-occurrence matrix is empty.")

    probability = cooccurrence / matrix_total
    row_masses = probability.sum(axis=1)
    column_masses = probability.sum(axis=0)
    expected_probability = np.outer(row_masses, column_masses)
    standardized_residuals = (
        (probability - expected_probability)
        / np.sqrt(np.outer(row_masses, column_masses))
    )

    left, singular_values, right_t = np.linalg.svd(standardized_residuals, full_matrices=False)
    eigenvalues = singular_values ** 2
    if len(eigenvalues) < 2:
        raise ValueError("At least two independent AFC dimensions are required.")

    row_coordinates = (left[:, :2] * singular_values[:2]) / np.sqrt(row_masses)[:, None]
    column_coordinates = (right_t[:2, :].T * singular_values[:2]) / np.sqrt(column_masses)[:, None]
    total_inertia = eigenvalues.sum()
    chi2_statistic = matrix_total * total_inertia
    explained = eigenvalues[:2] / total_inertia * 100
    return row_coordinates, column_coordinates, eigenvalues, explained, total_inertia, chi2_statistic, matrix_total


sector_names = sorted({label for labels in companies["sector_labels"] for label in labels})
sector_index = {label: index for index, label in enumerate(sector_names)}
cooccurrence = np.zeros((len(companies), len(sector_names)), dtype=float)

for row_index, labels in enumerate(companies["sector_labels"]):
    for label in labels:
        cooccurrence[row_index, sector_index[label]] = 1.0

sector_frequencies = cooccurrence.sum(axis=0)
active_columns = (sector_frequencies > 1) & (sector_frequencies < len(companies))
cooccurrence = cooccurrence[:, active_columns]
sector_frequencies = sector_frequencies[active_columns]
active_sector_names = np.asarray(sector_names)[active_columns]

active_rows = cooccurrence.sum(axis=1) > 0
companies_ca = companies.loc[active_rows].reset_index(drop=True)
cooccurrence = cooccurrence[active_rows]

(
    company_coordinates,
    sector_coordinates,
    eigenvalues,
    explained,
    total_inertia,
    chi2_statistic,
    matrix_total,
) = correspondence_analysis(cooccurrence)

afc_summary = pd.DataFrame(
    {
        "metric": [
            "enterprises",
            "active_sectors",
            "cooccurrences",
            "total_chi2",
            "total_inertia",
            "axis_1_eigenvalue",
            "axis_2_eigenvalue",
            "axis_1_inertia_pct",
            "axis_2_inertia_pct",
        ],
        "value": [
            cooccurrence.shape[0],
            cooccurrence.shape[1],
            matrix_total,
            chi2_statistic,
            total_inertia,
            eigenvalues[0],
            eigenvalues[1],
            explained[0],
            explained[1],
        ],
    }
)

print(f"AFC co-occurrence matrix: {cooccurrence.shape[0]} enterprises x {cooccurrence.shape[1]} active sectors")
print(f"Total co-occurrences: {matrix_total:.0f}")
print(f"Total chi-square: {chi2_statistic:,.2f}")
print(f"Total inertia: {total_inertia:.6f}")
print(f"Explained inertia: Axis 1 = {explained[0]:.1f}%, Axis 2 = {explained[1]:.1f}%")
display(afc_summary)

In [ ]:
company_export = companies_ca[["id", "name", "country", "capitalization_millions", "sector"]].copy()
company_export["axis_1"] = company_coordinates[:, 0]
company_export["axis_2"] = company_coordinates[:, 1]
company_export["label_size"] = min_max_scale(
    np.sqrt(company_export["capitalization_millions"].to_numpy()), 9, 22
)

sector_export = pd.DataFrame(
    {
        "sector": active_sector_names,
        "frequency": sector_frequencies.astype(int),
        "axis_1": sector_coordinates[:, 0],
        "axis_2": sector_coordinates[:, 1],
    }
).sort_values("frequency", ascending=False).reset_index(drop=True)
sector_export["point_size"] = min_max_scale(np.sqrt(sector_export["frequency"].to_numpy()), 13, 30)

company_export.to_csv(EXPORTS_DIR / "sector_ca_company_coordinates.csv", index=False, encoding="utf-8")
sector_export.to_csv(EXPORTS_DIR / "sector_ca_sector_coordinates.csv", index=False, encoding="utf-8")
afc_summary.to_csv(EXPORTS_DIR / "sector_ca_afc_summary.csv", index=False, encoding="utf-8")

print(f"Company coordinates: {EXPORTS_DIR / 'sector_ca_company_coordinates.csv'}")
print(f"Sector coordinates: {EXPORTS_DIR / 'sector_ca_sector_coordinates.csv'}")
print(f"AFC summary: {EXPORTS_DIR / 'sector_ca_afc_summary.csv'}")
display(sector_export[["sector", "frequency", "axis_1", "axis_2"]])

In [ ]:
figure = go.Figure()

figure.add_trace(
    go.Scatter(
        x=company_export["axis_1"],
        y=company_export["axis_2"],
        mode="markers+text",
        name="Enterprises",
        text=company_export["name"],
        textposition="top center",
        textfont={"size": company_export["label_size"].tolist(), "color": "#2E5280"},
        marker={"size": 7, "color": "#2E5280", "opacity": 0.55},
        customdata=np.column_stack(
            [
                company_export["capitalization_millions"],
                company_export["country"].fillna("Unknown"),
                company_export["sector"],
            ]
        ),
        hovertemplate=(
            "<b>%{text}</b><br>"
            "Capitalization: $%{customdata[0]:,.0f} M<br>"
            "Country: %{customdata[1]}<br>"
            "Sectors: %{customdata[2]}<extra></extra>"
        ),
    )
)

figure.add_trace(
    go.Scatter(
        x=sector_export["axis_1"],
        y=sector_export["axis_2"],
        mode="markers+text",
        name="Sectors",
        text=sector_export["sector"],
        textposition="bottom center",
        textfont={"size": 12, "color": "#8B4A52"},
        marker={
            "size": sector_export["point_size"],
            "color": "#B8727A",
            "line": {"color": "#8B4A52", "width": 1},
            "opacity": 0.9,
        },
        customdata=sector_export[["frequency"]],
        hovertemplate="<b>%{text}</b><br>Enterprise frequency: %{customdata[0]}<extra></extra>",
    )
)

figure.add_hline(y=0, line={"color": "#D5CDC5", "width": 1})
figure.add_vline(x=0, line={"color": "#D5CDC5", "width": 1})
figure.update_layout(
    title={
        "text": f"Sector Positioning Map - AFC (chi2 = {chi2_statistic:,.1f})",
        "font": {"color": "#516B0E"},
    },
    paper_bgcolor="#F2EDE4",
    plot_bgcolor="#FDFAF4",
    font={"family": "Inter, Segoe UI, sans-serif", "color": "#28241E"},
    xaxis={"title": f"AFC axis 1 ({explained[0]:.1f}% inertia)", "zeroline": False},
    yaxis={"title": f"AFC axis 2 ({explained[1]:.1f}% inertia)", "zeroline": False},
    legend={"orientation": "h", "y": 1.08, "x": 0},
    hovermode="closest",
    margin={"l": 50, "r": 50, "t": 90, "b": 80},
)
figure.add_annotation(
    text=(
        f"AFC on the enterprise x sector co-occurrence matrix. chi2 = {chi2_statistic:,.2f}; "
        "enterprise label size follows capitalization; sector point size follows frequency."
    ),
    xref="paper", yref="paper", x=0, y=-0.15, showarrow=False,
    font={"size": 11, "color": "#7A6E67"}, align="left",
)

map_path = EXPORTS_DIR / "sector_positioning_ca.html"
figure.write_html(map_path, include_plotlyjs="cdn")
figure.show()
print(f"Interactive map: {map_path}")

In [ ]:
from IPython.display import Image, display

png_path = EXPORTS_DIR / "sector_positioning_ca.png"
figure.write_image(png_path, width=1600, height=1000, scale=2)
print(f"Static image: {png_path}")
display(Image(filename=png_path))